# Reference Checker

Hypothesis: a hallucinated reference will (a) produce a title of a paper that doesn't exist, (b) make up authors for paper titles that do exist.

This script pulls the references section out of a PDF, pulls the references, and attempts to verify each title against Semantic Scholar.
If it finds the title in Semantic Scholar, it then attempts to verify each author, according to Semantic Scholar's records, appears in the reference.

It is really hard to deal with all reference formats, but also idosycracies and casual mistakes. This script attempts to find a title by looking for a span of dictionary-recognizable words under the assumption that names do not make for long spans of dictionary-recognizable words. The algorithm is allowed to see a maximum of one non-dictionary token in a row before concluding that a span is not the title. If there is more than one span that could be a title, it will pick the longest.

This algorithm will produce a lot of false alarms where it simply fails to pull the title out of reference.

**To Use:**

Run `check_refs(filepath)`

**Notes:**
- Some PDFs that will be reviewed have line numbers. The line numbers get interjected into the middle of text spans. the `pdf_has_line_numbers=True` option will remove all numbers from references. This shouldn't matter if the pdf has line numbers or not because the algorithm should already ignore dates.
- Add words that are not in the dictionary to `CUSTOM_VOCAB`.
- Add words that you expect never to be in a paper title to `FILTER`.
- Doesn't handle names with accent marks.
- Sometimes the dictionary decides that names are all words and when there are a lot of these in a row, it will pick this over a short title.
- Sometimes reference span page breaks, in which case you get some false alarms.
- Paper headers and footers get interjected with references and create false alarms.
- I should really check for names that are in the reference line but not in Semantic Scholar, but that's a pain because I would need to figure out what names are. I'm hoping that missing authors are a reasonable flag for possibility of hallucination.

**TODO:**
- Need to implement a reverse check of author names, to see if names in the reference are also in the authors json from Semantic Scholar. I've been avoiding trying to figure out what a name is. Probably everything that is before the title. Maybe I can then figure out last names?
- Send the entire reference line to Google Scholar search and see if the first hit is identical. How do I know it's identical? I guess the title would be a span in the reference? Then I think I could also get author names from Google Scholar's search page. 

# Install Packages

In [1]:
!pip install pymupdf4llm

In [2]:
!pip install spacy

In [3]:
!pip install PyEnchant

In [ ]:
!pip install pypdf

# Imports

In [157]:
import pymupdf4llm
import re
import enchant
import spacy
import unidecode
import string
import requests
import time
from pathlib import Path
from pypdf import PdfReader, PdfWriter
from functools import reduce
# Load the English language model
NLP = spacy.load("en_core_web_sm")
DICTIONARY = enchant.Dict("en_US")

# Globals

In [5]:
SEMANTIC_SCHOLAR_URL = 'https://api.semanticscholar.org/graph/v1/paper/search/match?query=' 

In [85]:
CUSTOM_VOCAB = ['ai', 'xai', 'operationalizing', 'seamful', 'llm', 'llms', 'vs']

In [7]:
FILTER = ['proceedings', 'conference',
          'NY', 'USA']

In [101]:
REPLACEMENTS = {'vs.': 'versus'}

# Helpers

In [8]:
def tokenize(text):
    return re.findall(r"\w+|[^\w\s-]", unidecode.unidecode(text))

In [9]:
def detokenize(tokens):
    result = ''
    for token in tokens:
        if token in string.punctuation:
            result = result + token
        else:
            result = result + ' ' + token
    return result.strip()

In [10]:
def is_number(s):
    try:
        float(s)
        return True
    except ValueError:
        return False

In [11]:
def remove_numbers(text):
    return re.sub(r'\d+', '', text)

In [12]:
def does_contain(word_list, targets):
    return reduce(lambda a, b: a | b, 
                  map(lambda t: t.lower() in [w.lower() for w in word_list], 
                      targets))

In [13]:
def remove_hanging_punctuation(word_list):
    if word_list[-1] in string.punctuation:
        return word_list[0:-1]
    else:
        return word_list

# Get Reference Section 

In [64]:
def get_ref_section(md_text):
    m = re.search(r'# [0-9 ]*\*\*References\*\*([a-zA-Z0-9 \(\)\.\,\;]*)', md_text)
    if m is not None:
        after = md_text[m.span()[1]:]
        m = re.search(r'# \*\*', after)
        if m is not None:
            return after[0:m.span()[0]].strip()
        else:
            return after
    else:
        return None

# Extract Title

Each reference is on its own line. Each reference is further broken into a list of tokens (words, punctuation)

In [104]:
def extract_title(tokens, verbose=False):
    candidates = [] # Candidate titles, longest preferred
    title = [] # Current title we are building
    skip = False # We get one skip in a row
    # Iterate through tokens
    for token in tokens:
        # Part of speech tagging
        doc = NLP(token)
        if verbose:
            print(">>", token)
        # If we get a hash or star, we crash out
        # if token in ['#', '*']:
        #     return None
        # If we get certain punctuation we finish the title building
        if token in ['.', ';', '[', ']', '(', ')']:
            if verbose:
                print("PUNCT")  
                print("TITLE=", title)  
            # If title is 4 or more, then we keep it
            if len(title) > 3 and title[0].lower() != 'in':
                if verbose:
                    print("CANDIDATE FOUND")
                candidates.append(title)
                title = []
                skip = False
            # If title is less than 4 we throw it out
            else:
                if verbose:
                    print("NOT A CANDIDATE")
                title = []
                skip = False
        # We cannot start a title with , or and or :
        elif (token == ',' or token == 'and' or token == ':') and len(title) == 0:
            if verbose:
                print("START WITH COMMA OR AND")
            title = []
            skip = False
        # We found something that is in the dictionaries, and is length greater than 1 (unless I or A) and is not "and"
        elif (DICTIONARY.check(token) or token.lower() in CUSTOM_VOCAB) and (len(token) > 1 or token == 'I' or token.lower() == 'a') and token.lower != 'and':
            title.append(token)
            skip = False
            if verbose:
                print("TOK")
        # Whatever remains is probably okay, but we use a skip
        elif not skip:
            skip = True
            title.append(token)
            if verbose:
                print("TOK+SKIP")
        # If we are here, we are on our second skip, give up on this
        else:
            title = []
            skip = False
            if verbose:
                print("SKIP") 
    # Now we filter out candidates
    filtered_candidates = list(filter(lambda title: not does_contain(title, FILTER), candidates)) 
    # remove orphaned punctuation
    filtered_candidates = list(map(lambda title: remove_hanging_punctuation(title),
                                   filtered_candidates))
    if len(filtered_candidates) > 0:
        sorted_candidates = sorted(filtered_candidates, key=len, reverse=True)
        return list(filter(lambda token: not is_number(token), sorted_candidates[0]))
    else:
        return None

# Access Semantic Scholar

An `author` is a json structure.

In [16]:
def get_authors_from_semantic_scholar(title):
    url = SEMANTIC_SCHOLAR_URL + title
    query_params = {"fields": "title,authors"}
    headers = {}
    response = requests.get(url, params=query_params, headers=headers)
    if response.status_code == 200:
        response_data = response.json()
        return response_data['data'][0]['authors']
    else:
        return None

In [93]:
def check_author(ref, author):
    last_name = unidecode.unidecode(author['name'].split()[-1]).lower()
    return last_name in ref

def check_authors(ref, authors):
    ref = unidecode.unidecode(ref).lower()
    success = True
    for author in authors:
        if not check_author(ref, author):
            print("AUTHOR", author['name'], "NOT FOUND")
            success = False
    return success

# Check References

`pdf_has_line_numbers=True` will remove all numbers from each reference line.

In [168]:
def crop_pdf(filename, left_margin):
    reader = PdfReader("tests/tda.pdf")
    writer = PdfWriter()
    path = Path("tests/data.txt")
    new_path = path.parent / (path.stem + "_cropped" + path.suffix)
    for page in reader.pages:
        # Get current crop box (or media box if no crop box set)
        box = page.cropbox        
        # Shift the left edge inward by 1 inch
        page.cropbox.left = box.left + left_margin
        writer.add_page(page)
    with open(new_path, "wb") as f:
        writer.write(f)
    return new_path

In [169]:
def check_refs(filename, sleep=30, pdf_has_line_numbers = False, left_margin = 0):
    # If line numbers, strip the left margin off
    if pdf_has_line_numbers:
        left_margin = max(72*0.75, left_margin) # 72 points is one inch
    if left_margin > 0:
        print("Cropping...")
        filename = crop_pdf(filename, left_margin)
    # Convert PDF to markdown
    print("Converting PDF to markdown...")
    md_text = pymupdf4llm.to_markdown(filename, header=False, footer=False)
    # Remove numbers if the PDF has line numbers
    # if pdf_has_line_numbers:
    #     md_text = remove_numbers(md_text)
    four_newline_check = re.search(r'(\n[ \t]*){4,}', ref_section)
    if four_newline_check is not None:
        refs = [r.strip() for r in re.split(r'(\n[ \t]*){4,}', ref_section.replace("_", "")) if r.strip()]
    else:
        refs = [r.strip() for r in ref_section.replace("_", "").split("\n\n") if r.strip()]
    # get references section, split into lines
    # refs = get_ref_section(md_text).strip().replace('_', '').split('\n\n')
    # Each reference should now be a separate string in a list
    print("Checking", len(refs), "refs...")
    # Iterate through each reference line
    for n, ref in enumerate(refs):
        print(n)
        ref = ref.replace('\n','')
        for key in REPLACEMENTS:
            ref = ref.replace(key, REPLACEMENTS[key])
        # Get the title
        title = extract_title(tokenize(ref))
        # If title is found, check the authors
        if title is not None and len(title) > 0:
            # De-tokenize the title to get ready for Semantic Scholar search
            title = detokenize(title)
            print(title)
            # search semantic scholar and bring back a data record including authors
            authors = get_authors_from_semantic_scholar(title)
            # If authors are found then the Semantic Scholar search succeeded
            if authors is not None:
                print("FOUND in Semantic Scholar")
                # check authors
                if check_authors(ref, authors):
                    print("OK")
            # Semantic Scholar search failed
            else:
                print("NOT FOUND in Semantic Scholar")
            # Sleep
            time.sleep(sleep)
        # No title extracted
        else:
            # Report the raw text
            print(ref)
            print('NO TITLE FOUND')
        print('\n')
        

# Run Me

In [ ]:
check_refs("tests/tda.pdf", pdf_has_line_numbers = True, sleep=30)

Cropping...
Converting PDF to markdown...
=== Document parser messages ===
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

# For Testing

In [127]:
filename = "tests/tda.pdf"
md_text = pymupdf4llm.to_markdown(filename, write_images=False, header=False, footer=False)

=== Document parser messages ===
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [129]:
md_text

'000 001 002 003 004 005 006 007 008 009 010 **Abstract** 011 012 We use training-data attribution as a mechanistic013 discovery tool to ask which regions of the pretrain014 ing corpus support social-reasoning versus STEM015 reasoning capability in OLMo3-7B. Training-data 016 attribution measures how strongly each training 017 document influences a model’s predictions on a 018 benchmark, but document-level scores are too 019 noisy to identify which corpus regions support 020 which capabilities, and prior work has empha021 sized factual knowledge rather than reasoning. 022 We compute gradient-based attribution (TrackStar 023 via Bergson) over a working set drawn from the 024 de-duplicated Dolma3 mix, aggregate influence 025 across WebOrganizer’s 24-format _×_ 24-topic tax026 onomy (576 bins), and contrast benchmark pairs 027 in a 2 _×_ 2 design that varies domain (social vs. 028 STEM) and capability type (reasoning vs. knowl029 edge): SocialIQA and MMLU Social Sciences 030 against ARC-C

In [111]:
ref_section = get_ref_section(md_text)
ref_section

'-  \n\n-  ¨ Akyurek, E., Bolukbasi, T., Liu, F., Xiong, B., Tenney, I., \n\n-  Andreas, J., and Guu, K. Towards tracing knowledge in \n\n-  language models back to the training data. In _Findings of_ \n\n-  _the Association for Computational Linguistics: EMNLP_ \n\n-  __ , pp. –, Abu Dhabi, United Arab Emi- \n\n-  rates, December . Association for Computational \n\n-  Linguistics. doi: ./v/.findings-emnlp. \n\n-  . URL https://aclanthology.org/. \n\n-  findings-emnlp./. \n\n-  \n\n-  Antoniades, A., Wang, X., Elazar, Y., Amayuelas, A., Al balak, A., Zhang, K., and Wang, W. Y. Generalization  vs. memorization: Tracing language models’ capabili ties back to pretraining data. In _ICML  Workshop_  _on Foundation Models in the Wild_ , . URL https:  //openreview.net/forum?id=LaybrPql. \n\n-  \n\n-  Basu, S., Pope, P., and Feizi, S. Influence functions in  deep learning are fragile. In _International Conference_  _on Learning Representations_ , . URL https://  openreview.net/forum?id=xHKVVHG

In [117]:
refs[2]

'Antoniades, A., Wang, X., Elazar, Y., Amayuelas, A., Al balak, A., Zhang, K., and Wang, W. Y. Generalization  vs. memorization: Tracing language models’ capabili ties back to pretraining data. In ICML  Workshop  on Foundation Models in the Wild , . URL https:  //openreview.net/forum?id=LaybrPql.'

In [106]:
ref = refs[2]
for key in REPLACEMENTS:
    ref = ref.replace(key, REPLACEMENTS[key])
extract_title(tokenize(ref), verbose=True)

>> Antoniades
TOK+SKIP
>> ,
SKIP
>> A
TOK
>> .
PUNCT
TITLE= ['A']
NOT A CANDIDATE
>> ,
START WITH COMMA OR AND
>> Wang
TOK
>> ,
TOK+SKIP
>> X
SKIP
>> .
PUNCT
TITLE= []
NOT A CANDIDATE
>> ,
START WITH COMMA OR AND
>> Elazar
TOK+SKIP
>> ,
SKIP
>> Y
TOK+SKIP
>> .
PUNCT
TITLE= ['Y']
NOT A CANDIDATE
>> ,
START WITH COMMA OR AND
>> Amayuelas
TOK+SKIP
>> ,
SKIP
>> A
TOK
>> .
PUNCT
TITLE= ['A']
NOT A CANDIDATE
>> ,
START WITH COMMA OR AND
>> Al
TOK
>> balak
TOK+SKIP
>> ,
SKIP
>> A
TOK
>> .
PUNCT
TITLE= ['A']
NOT A CANDIDATE
>> ,
START WITH COMMA OR AND
>> Zhang
TOK+SKIP
>> ,
SKIP
>> K
TOK+SKIP
>> .
PUNCT
TITLE= ['K']
NOT A CANDIDATE
>> ,
START WITH COMMA OR AND
>> and
START WITH COMMA OR AND
>> Wang
TOK
>> ,
TOK+SKIP
>> W
SKIP
>> .
PUNCT
TITLE= []
NOT A CANDIDATE
>> Y
TOK+SKIP
>> .
PUNCT
TITLE= ['Y']
NOT A CANDIDATE
>> Generalization
TOK
>> versus
TOK
>> memorization
TOK
>> :
TOK+SKIP
>> Tracing
TOK
>> language
TOK
>> models
TOK
>> '
TOK+SKIP
>> capabili
SKIP
>> ties
TOK
>> back
TOK
>> to
TOK


['ties', 'back', 'to', 'pretraining', 'data']

In [148]:
four_newline_check = re.search(r'(\n[ \t-]*){4,}', ref_section)
if four_newline_check is not None:
    refs = [r.strip() for r in re.split(r'(\n[ \t-]*){4,}', ref_section.replace("_", "")) if r.strip()]
else:
    refs = [r.strip() for r in ref_section.replace("_", "").split("\n\n") if r.strip()]
refs = [r.replace('\n','') for r in refs]
refs

['- Akyurek, E., Bolukbasi, T., Liu, F., Xiong, B., Tenney, I.,¨ Andreas, J., and Guu, K. Towards tracing knowledge in language models back to the training data. In Findings of the Association for Computational Linguistics: EMNLP 2022 , pp. 2429–2446, Abu Dhabi, United Arab Emirates, December 2022. Association for Computational Linguistics. doi: 10.18653/v1/2022.findings-emnlp. 180. URL https://aclanthology.org/2022. findings-emnlp.180/.',
 '- Antoniades, A., Wang, X., Elazar, Y., Amayuelas, A., Albalak, A., Zhang, K., and Wang, W. Y. Generalization vs. memorization: Tracing language models’ capabilities back to pretraining data. In ICML 2024 Workshop on Foundation Models in the Wild , 2024. URL https: //openreview.net/forum?id=0LaybrPql4.',
 '- Basu, S., Pope, P., and Feizi, S. Influence functions in deep learning are fragile. In International Conference on Learning Representations , 2021. URL https:// openreview.net/forum?id=xHKVVHGDOEk.',
 '- Bender, E. M. and Friedman, B. Data stat